# NovaGen Research Labs

In [26]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [27]:
data = pd.read_csv("/kaggle/input/datasets/premraj25/novagen-research-labs/novagen_dataset.csv")

In [28]:
data.head()

,Age,BMI,Blood_Pressure,Cholesterol,Glucose_Level,Heart_Rate,Sleep_Hours,Exercise_Hours,Water_Intake,Stress_Level,...,Diet,MentalHealth,PhysicalActivity,MedicalHistory,Allergies,Diet_Type__Vegan,Diet_Type__Vegetarian,Blood_Group_AB,Blood_Group_B,Blood_Group_O
0,2.0,26.0,111.0,198.0,99.0,72.0,4.0,1.0,5.0,5.0,...,1,2,1,0,1,False,True,True,False,False
1,8.0,24.0,121.0,199.0,103.0,75.0,2.0,1.0,2.0,9.0,...,1,2,1,2,2,False,False,True,False,False
2,81.0,27.0,147.0,203.0,100.0,74.0,10.0,-0.0,5.0,1.0,...,2,0,0,1,0,True,False,False,False,False
3,25.0,21.0,150.0,199.0,102.0,70.0,7.0,3.0,3.0,3.0,...,1,2,1,2,0,True,False,False,True,False
4,24.0,26.0,146.0,202.0,99.0,76.0,10.0,2.0,5.0,1.0,...,2,0,2,0,2,False,True,False,True,False


In [29]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9549 entries, 0 to 9548
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Age                    9549 non-null   float64
 1   BMI                    9549 non-null   float64
 2   Blood_Pressure         9549 non-null   float64
 3   Cholesterol            9549 non-null   float64
 4   Glucose_Level          9549 non-null   float64
 5   Heart_Rate             9549 non-null   float64
 6   Sleep_Hours            9549 non-null   float64
 7   Exercise_Hours         9549 non-null   float64
 8   Water_Intake           9549 non-null   float64
 9   Stress_Level           9549 non-null   float64
 10  Target                 9549 non-null   int64  
 11  Smoking                9549 non-null   int64  
 12  Alcohol                9549 non-null   int64  
 13  Diet                   9549 non-null   int64  
 14  MentalHealth           9549 non-null   int64  
 15  Phys

In [30]:
data.isnull().sum()

Age                      0
BMI                      0
Blood_Pressure           0
Cholesterol              0
Glucose_Level            0
Heart_Rate               0
Sleep_Hours              0
Exercise_Hours           0
Water_Intake             0
Stress_Level             0
Target                   0
Smoking                  0
Alcohol                  0
Diet                     0
MentalHealth             0
PhysicalActivity         0
MedicalHistory           0
Allergies                0
Diet_Type__Vegan         0
Diet_Type__Vegetarian    0
Blood_Group_AB           0
Blood_Group_B            0
Blood_Group_O            0
dtype: int64

In [31]:
Y = data["Target"]
Y.head()

0    1
1    1
2    0
3    0
4    0
Name: Target, dtype: int64

In [32]:
# Data encoding 

col = ["Diet_Type__Vegan","Diet_Type__Vegetarian", "Blood_Group_AB", "Blood_Group_B", "Blood_Group_O"]
for x in col:
    data[x] = data[x].map({True: 1, False: 0})

X = data.drop(columns=["Target"] , axis=1)

In [33]:
X.head()

,Age,BMI,Blood_Pressure,Cholesterol,Glucose_Level,Heart_Rate,Sleep_Hours,Exercise_Hours,Water_Intake,Stress_Level,...,Diet,MentalHealth,PhysicalActivity,MedicalHistory,Allergies,Diet_Type__Vegan,Diet_Type__Vegetarian,Blood_Group_AB,Blood_Group_B,Blood_Group_O
0,2.0,26.0,111.0,198.0,99.0,72.0,4.0,1.0,5.0,5.0,...,1,2,1,0,1,0,1,1,0,0
1,8.0,24.0,121.0,199.0,103.0,75.0,2.0,1.0,2.0,9.0,...,1,2,1,2,2,0,0,1,0,0
2,81.0,27.0,147.0,203.0,100.0,74.0,10.0,-0.0,5.0,1.0,...,2,0,0,1,0,1,0,0,0,0
3,25.0,21.0,150.0,199.0,102.0,70.0,7.0,3.0,3.0,3.0,...,1,2,1,2,0,1,0,0,1,0
4,24.0,26.0,146.0,202.0,99.0,76.0,10.0,2.0,5.0,1.0,...,2,0,2,0,2,0,1,0,1,0


In [34]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,Y,
    test_size=0.3, random_state=42
)

In [35]:
# Data Scaling 

from sklearn.preprocessing import StandardScaler

scale = StandardScaler()

X_train = scale.fit_transform(X_train)
X_test = scale.transform(X_test)

## Heterogeneous ensemble

### Voting Classifier

In [36]:
# Models

from sklearn.ensemble import VotingClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier


lr = LogisticRegression(max_iter=2000)
svc = SVC()
dtc = DecisionTreeClassifier(max_depth=4)
knn = KNeighborsClassifier()

votclf = VotingClassifier(
    estimators=[
        ("Logistic Regression", lr),
        ("SVC", svc),
        ("Decision Tree", dtc),
        ("KNN",knn)
    ]
)

In [37]:
votclf.fit(X_train,Y_train)

VotingClassifier(estimators=[('Logistic Regression',
                              LogisticRegression(max_iter=2000)),
                             ('SVC', SVC()),
                             ('Decision Tree',
                              DecisionTreeClassifier(max_depth=4)),
                             ('KNN', KNeighborsClassifier())])

In [38]:
Y_pred = votclf.predict(X_test)
Y_pred_t = votclf.predict(X_train)

print(f"Accuracy score for train data: {accuracy_score(Y_train,Y_pred_t)*100} %")
print(f"Accuracy score for test data: {accuracy_score(Y_test,Y_pred)*100} %")

Accuracy score for train data: 92.01077199281868 %
Accuracy score for test data: 89.42408376963351 %


### Stacking Classifier

In [39]:
# Models

from sklearn.ensemble import StackingClassifier

meta_model = KNeighborsClassifier()

staclf = StackingClassifier(
    estimators=[
        ("Logistic Regression", lr),
        ("SVC", svc),
        ("Decision Tree", dtc),
        ("KNN",knn)
    ],
    final_estimator = meta_model,
    cv=5
)

In [40]:
staclf.fit(X_train,Y_train)

StackingClassifier(cv=5,
                   estimators=[('Logistic Regression',
                                LogisticRegression(max_iter=2000)),
                               ('SVC', SVC()),
                               ('Decision Tree',
                                DecisionTreeClassifier(max_depth=4)),
                               ('KNN', KNeighborsClassifier())],
                   final_estimator=KNeighborsClassifier())

In [41]:
Y_pred = staclf.predict(X_test)
Y_pred_t = staclf.predict(X_train)

print(f"Accuracy score for train data: {accuracy_score(Y_train,Y_pred_t)*100} %")
print(f"Accuracy score for test data: {accuracy_score(Y_test,Y_pred)*100} %")

Accuracy score for train data: 95.22740873728306 %
Accuracy score for test data: 92.77486910994764 %


## Homogeneous ensemble

### Random Forest Classifier (Bagging)

In [42]:
# Models

from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier(
    n_estimators=200,
    oob_score=True,
    max_depth=6,
    random_state=42
)

In [43]:
rfc.fit(X_train,Y_train)

RandomForestClassifier(max_depth=6, n_estimators=200, oob_score=True,
                       random_state=42)

In [44]:
Y_pred = rfc.predict(X_test)
Y_pred_t = rfc.predict(X_train)

print(f"Accuracy score for train data: {accuracy_score(Y_train,Y_pred_t)*100} %")
print(f"Accuracy score for test data: {accuracy_score(Y_test,Y_pred)*100} %")

Accuracy score for train data: 90.27528426092161 %
Accuracy score for test data: 89.84293193717278 %


### Gradient Boosting Classifier (Boosting)

In [45]:
# Models

from sklearn.ensemble import GradientBoostingClassifier

gbc = GradientBoostingClassifier(
    n_estimators=250,
    subsample=0.8,
    max_depth = 4,
    random_state=42
)

In [46]:
gbc.fit(X_train,Y_train)

GradientBoostingClassifier(max_depth=4, n_estimators=250, random_state=42,
                           subsample=0.8)

In [47]:
Y_pred = gbc.predict(X_test)
Y_pred_t = gbc.predict(X_train)

print(f"Accuracy score for train data: {accuracy_score(Y_train,Y_pred_t)*100} %")
print(f"Accuracy score for test data: {accuracy_score(Y_test,Y_pred)*100} %")

Accuracy score for train data: 97.84560143626571 %
Accuracy score for test data: 94.65968586387434 %


### AdaBoost Classifier (Boosting)

In [51]:
# Models

from sklearn.ensemble import AdaBoostClassifier

base_model = DecisionTreeClassifier(
    max_depth=2
)

adc = AdaBoostClassifier(
    estimator = base_model,
    random_state = 42,
    n_estimators = 500
)

In [52]:
adc.fit(X_train,Y_train)

AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2),
                   n_estimators=500, random_state=42)

In [53]:
Y_pred = adc.predict(X_test)
Y_pred_t = adc.predict(X_train)

print(f"Accuracy score for train data: {accuracy_score(Y_train,Y_pred_t)*100} %")
print(f"Accuracy score for test data: {accuracy_score(Y_test,Y_pred)*100} %")

Accuracy score for train data: 93.58168761220826 %
Accuracy score for test data: 93.40314136125654 %
